In [ ]:
##!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

import numpy as np
import matplotlib

import matplotlib.pyplot as plt

import time
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.cluster import KMeans
import umap
from sklearn.cluster import DBSCAN
from sklearn import metrics

from tqdm import tqdm_notebook
from lmfit import minimize, Parameters
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as pl
import shap

import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import numpy as np

# --------- bandwidth helper (Silverman) ----------
def _silverman_sigma(a):
    a = np.asarray(a, float)
    n = a.size
    if n < 2:
        return 1.0
    std = np.nanstd(a)
    iqr = np.subtract(*np.percentile(a, [75, 25]))
    sigma = min(std, iqr / 1.349) or std or (np.nanmax(a) - np.nanmin(a) + 1e-9)
    return 1.06 * sigma * (n ** (-1/5))

def _ensure_edges(edges_or_centers, nbins, data_min, data_max):
    if edges_or_centers is None:
        return np.linspace(data_min, data_max, nbins + 1)
    arr = np.asarray(edges_or_centers, float)
    if arr.size == nbins + 1:
        return arr
    if arr.size == nbins:  # centers -> edges
        centers = arr
        step = np.diff(centers)
        step = np.r_[step[:1], step]  # assume first step repeats
        edges = np.r_[centers - step/2, centers[-1] + step[-1]/2]
        return edges
    raise ValueError("bins must be len=n or n+1 (centers or edges).")

def _gaussian_kernel1d(sigma_bins, truncate=4.0):
    if sigma_bins <= 0:
        return np.array([1.0])
    radius = int(max(1, np.floor(truncate * sigma_bins)))
    x = np.arange(-radius, radius + 1, dtype=float)
    k = np.exp(-0.5 * (x / sigma_bins) ** 2)
    k /= k.sum()
    return k

# --------------------------------------------------
def drevi_matrix(
    x, y,
    x_bins=None, y_bins=None,          # edges (n+1) or centers (n); if None, auto
    nx=120, ny=120,                    # used if bins are None
    hx=None, hy=None,                  # bandwidths in data units; if None, Silverman
    method="hist_gauss",               # "hist_gauss" (fast) or "kde" (Gaussian, compact)
    truncate=4.0,                      # kernel half-width in sigmas
    return_counts=False                # also return column counts before smoothing
):
    """
    Compute a DREVI matrix (rows=y, cols=x). Columns sum to 1 (p(y|x)).
    No kNN anywhere.

    method="hist_gauss":
        1) 2D histogram (counts) on (x,y) grid
        2) separable Gaussian smoothing along x and y
        3) column-normalize

    method="kde":
        For each x-bin center x_j, compute Gaussian weights in x (with cutoff),
        then Gaussian KDE in y using those weights; normalize column.

    Returns
    -------
    M : (ny, nx) float array   # conditional density p(y|x)
    x_edges, y_edges : arrays
    counts_x : (nx,) float array (only if return_counts=True)  # column mass pre-normalization
    """
    x = np.asarray(x, float).ravel()
    y = np.asarray(y, float).ravel()
    assert x.size == y.size, "x and y must have same length"
    n = x.size
    if n == 0:
        raise ValueError("Empty data")

    # bandwidths (data units)
    hx = float(hx) if hx is not None else _silverman_sigma(x)
    hy = float(hy) if hy is not None else _silverman_sigma(y)
    if hx <= 0: hx = max(1e-9, (np.nanmax(x)-np.nanmin(x))/1000.0)
    if hy <= 0: hy = max(1e-9, (np.nanmax(y)-np.nanmin(y))/1000.0)

    # bin edges (in data units)
    x_edges = _ensure_edges(x_bins, nx, np.nanmin(x), np.nanmax(x))
    y_edges = _ensure_edges(y_bins, ny, np.nanmin(y), np.nanmax(y))
    x_centers = (x_edges[:-1] + x_edges[1:]) / 2.0
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2.0

    if method == "hist_gauss":
        # 1) 2D histogram of counts
        H, xe, ye = np.histogram2d(x, y, bins=[x_edges, y_edges])
        H = H.T  # rows=y, cols=x
        counts_x = H.sum(axis=0)

        # 2) separable Gaussian smoothing in *bin units*
        # convert bandwidths (data) -> in bins
        sx = hx / np.mean(np.diff(x_edges))
        sy = hy / np.mean(np.diff(y_edges))
        kx = _gaussian_kernel1d(max(1e-6, sx), truncate=truncate)
        ky = _gaussian_kernel1d(max(1e-6, sy), truncate=truncate)

        # convolve along x (rows fixed), then along y (cols fixed)
        # (pure NumPy; SciPy's gaussian_filter would be faster if available)
        def _conv1d_same(mat, kernel, axis):
            k = kernel
            if axis == 1:  # along x
                out = np.empty_like(mat, dtype=float)
                for i in range(mat.shape[0]):
                    out[i] = np.convolve(mat[i], k, mode="same")
                return out
            else:          # along y
                out = np.empty_like(mat, dtype=float)
                for j in range(mat.shape[1]):
                    out[:, j] = np.convolve(mat[:, j], k, mode="same")
                return out

        S = _conv1d_same(H, kx, axis=1)
        S = _conv1d_same(S, ky, axis=0)

        # 3) column-normalize to p(y|x)
        col_sum = S.sum(axis=0, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            M = np.divide(S, col_sum, where=col_sum > 0)
            M[:, col_sum.ravel() == 0] = 0.0

        if return_counts:
            return M, x_edges, y_edges, counts_x
        return M, x_edges, y_edges

    elif method == "kde":
        # Precompute y-kernel matrix for fast reuse (with cutoff)
        dy = (y_centers[:, None] - y[None, :]) / hy
        mask_y = np.abs(dy) <= truncate
        Gy = np.zeros_like(dy, dtype=float)
        Gy[mask_y] = np.exp(-0.5 * dy[mask_y] ** 2)
        Gy /= (np.sqrt(2*np.pi) * hy)  # proper Gaussian pdf along y

        M = np.zeros((ny, nx), float)
        counts_x = np.zeros(nx, float)

        inv_sqrt2pi_hx = 1.0 / (np.sqrt(2*np.pi) * hx)
        for j, xc in enumerate(x_centers):
            dx = (x - xc) / hx
            keep = np.abs(dx) <= truncate
            if not np.any(keep):
                continue
            wx = np.exp(-0.5 * dx[keep] ** 2) * inv_sqrt2pi_hx  # (m,)
            # KDE in y with weights wx
            col = Gy[:, keep] @ wx  # (ny,)
            s = col.sum()
            if s > 0:
                M[:, j] = col / s
                counts_x[j] = wx.sum()
            else:
                M[:, j] = 0.0
                counts_x[j] = 0.0

        if return_counts:
            return M, x_edges, y_edges, counts_x
        return M, x_edges, y_edges

    else:
        raise ValueError("method must be 'hist_gauss' or 'kde'")


def plot_drevi_with_mean(
    density,                  # 2D array: (n_y, n_x), columns ~ weights for p(y|x)
    x_bins=None,              # edges (n_x+1) or centers (n_x)
    y_bins=None,              # edges (n_y+1) or centers (n_y)
    smooth_sigma=1.5,         # smoothing in *x-bin units*; 0 disables
    ci=0.95,                  # confidence level in (0,1) or 0 to disable band
    counts_per_col=None,      # optional effective sample size per column (len=n_x)
    cmap="viridis",
    ax=None,
    line_kwargs=None,
    band_kwargs=None,
    imshow_kwargs=None,tit=None,xlab='X',ylab='Y'
):
    """
    Returns
    -------
    ax, mean_y, mean_y_smooth, (lower_smooth, upper_smooth)
    Notes
    -----
    - If counts_per_col is None:
        * If columns look like counts (sum > 1), uses column sums as n.
        * If columns sum to ~1 (pure densities), falls back to an SD band
          (i.e., uses n=1 -> not a true CI, but a variability band).
    """
    M = np.asarray(density)
    if M.ndim != 2:
        raise ValueError("density must be a 2D array (n_y, n_x)")
    ny, nx = M.shape

    line_kwargs = {"lw": 2, "color": "white"} | (line_kwargs or {})
    band_color = line_kwargs.get("color", "white")
    band_kwargs = {"alpha": 0.25, "facecolor": band_color, "edgecolor": "none"} | (band_kwargs or {})
    imshow_kwargs = {"origin": "lower", "aspect": "auto", "cmap": cmap} | (imshow_kwargs or {})

    # --- helper: accept edges or centers
    def _to_centers(arr, n, name):
        if arr is None:
            return np.arange(n, dtype=float), None
        arr = np.asarray(arr, dtype=float)
        if arr.size == n + 1:      # edges
            return (arr[:-1] + arr[1:]) / 2.0, arr
        if arr.size == n:          # centers
            return arr, None
        raise ValueError(f"{name} must have length {n} (centers) or {n+1} (edges)")

    x_centers, x_edges = _to_centers(x_bins, nx, "x_bins")
    y_centers, y_edges = _to_centers(y_bins, ny, "y_bins")

    # --- weighted mean & variance per column
    col_mass = M.sum(axis=0)                                  # shape (nx,)
    with np.errstate(invalid="ignore", divide="ignore"):
        mean_y = (M.T @ y_centers) / col_mass                 # (nx,)
        mean_y2 = (M.T @ (y_centers**2)) / col_mass           # E[y^2 | x]
    var_y = mean_y2 - mean_y**2
    var_y[var_y < 0] = 0.0                                    # numeric guard

    # --- effective n per column (for SE and CI)
    # If user provides counts, use them; else:
    # - if col sums > 1, treat as counts; else use n=1 (SD band, not true CI)
    if counts_per_col is not None:
        n_eff = np.asarray(counts_per_col, dtype=float)
        if n_eff.shape != (nx,):
            raise ValueError("counts_per_col must have length n_x")
    else:
        n_eff = np.where(col_mass > 1.0, col_mass, 1.0)

    with np.errstate(invalid="ignore", divide="ignore"):
        se_y = np.sqrt(var_y / n_eff)

    # --- build Gaussian kernel & smoothing (cap kernel so len <= nx)
    def _smooth_1d(arr, sigma_bins):
        arr = np.asarray(arr, dtype=float)
        if not (sigma_bins and sigma_bins > 0) or nx < 3:
            return arr.copy()
        half_nominal = int(np.ceil(6 * float(sigma_bins)))
        half_cap = max(1, (nx - 1) // 2)
        half = min(max(3, half_nominal), half_cap)
        kx = np.arange(-half, half + 1, dtype=float)
        kernel = np.exp(-0.5 * (kx / float(sigma_bins)) ** 2)
        kernel /= kernel.sum()
        mask_nan = ~np.isfinite(arr)
        fill_val = np.nanmean(arr)
        if not np.isfinite(fill_val):
            fill_val = 0.0
        tmp = np.where(mask_nan, fill_val, arr)
        out = np.convolve(tmp, kernel, mode="same")   # length = nx
        out[mask_nan] = np.nan
        return out

    mean_y_smooth = _smooth_1d(mean_y, smooth_sigma)
    se_y_smooth   = _smooth_1d(se_y,   smooth_sigma)

    # --- CI band
    if ci and (0 < ci < 1):
        # two-sided z for CI level
        from math import sqrt
        # approximate z without SciPy:
        # common z for 95% ~ 1.96, 90% ~ 1.645, 68% ~ 1.0
        CI2Z = {0.68: 1.0, 0.90: 1.6448536269514722, 0.95: 1.959963984540054}
        z = CI2Z.get(round(ci, 2), 1.959963984540054 if ci > 0.9 else 1.0)
        halfwidth = z * se_y_smooth
        lower_smooth = mean_y_smooth - halfwidth
        upper_smooth = mean_y_smooth + halfwidth
    else:
        lower_smooth = upper_smooth = None

    # --- plot heatmap + line + band
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))

    if x_edges is not None and y_edges is not None:
        extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
        im = ax.imshow(M, extent=extent, **imshow_kwargs)
        if lower_smooth is not None:
            ax.fill_between(x_centers, lower_smooth, upper_smooth, **band_kwargs)
        ax.plot(x_centers, mean_y_smooth, **line_kwargs)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
    else:
        im = ax.imshow(M, **imshow_kwargs)
        x_pos = np.arange(nx)
        if y_bins is None:
            if lower_smooth is not None:
                ax.fill_between(x_pos, lower_smooth, upper_smooth, **band_kwargs)
            ax.plot(x_pos, mean_y_smooth, **line_kwargs)
            ax.set_ylabel(ylab)
        else:
            # if y is in value units but imshow uses indices, map Y->row index for overlay
            y_to_row = np.interp(mean_y_smooth, y_centers, np.arange(ny))
            if lower_smooth is not None:
                low_idx  = np.interp(lower_smooth, y_centers, np.arange(ny))
                high_idx = np.interp(upper_smooth, y_centers, np.arange(ny))
                ax.fill_between(x_pos, low_idx, high_idx, **band_kwargs)
            ax.plot(x_pos, y_to_row, **line_kwargs)
            ax.set_ylabel("y bin (index)")
        ax.set_xlabel(xlab)

    cbar = plt.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label("density")
    ax.set_title(tit)

    return ax, mean_y, mean_y_smooth, (lower_smooth, upper_smooth)


# Load and initialize

In [ ]:
import glob

In [ ]:
dir="/Users/ronguy/Dropbox/CyTOF_Breast/PDX/202403_lum_PDX/"
params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
FList=glob.glob(dir+"*")

In [ ]:
FList.sort()
FList

In [ ]:
DBs=['PDX5','PDX6','PDX7','PDX8']

In [ ]:
DBs

In [ ]:
Rep=dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:,:].values)

In [ ]:
for F,DB in zip(FList,DBs):
    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)

In [ ]:
N=list(globals()[DBs[0]].columns)
N.sort()
N

In [ ]:
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)

In [ ]:
NamesAll.sort()
NormMRK.sort()
EpiCols.sort()


In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
len(NamesAll)

In [ ]:
for DB in DBs:
    
    globals()[DB]=globals()[DB][NamesAll]


In [ ]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

def select_cd298_high_mhc_low(
    df: pd.DataFrame, cd298_col="CD298", mhc_col="MHC", return_mask=False, random_state=0
):
    """
    Returns the index of rows belonging to the cluster with high CD298 and low MHC.
    Uses a 2-component Gaussian Mixture on standardized features.
    """
    # keep only rows with both values present
    Xdf = df[[cd298_col, mhc_col]].dropna()
    X = Xdf.to_numpy()

    # standardize for stable clustering
    scaler = StandardScaler()
    Xz = scaler.fit_transform(X)

    # fit 2-component GMM
    gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=random_state)
    labels = gmm.fit_predict(Xz)

    # identify the component that is CD298-high & MHC-low
    means = gmm.means_              # in z-score space: columns [CD298z, MHCz]
    # Compute a score: +CD298z - MHCz (bigger means higher CD298 and lower MHC)
    scores = means[:, 0] - means[:, 1]
    target_comp = np.argmax(scores)

    # map back to the original dataframe's index (note we dropped NaNs above)
    selected_mask = (labels == target_comp)
    selected_index = Xdf.index[selected_mask]

    if return_mask:
        # full-length mask aligned to df
        full_mask = pd.Series(False, index=df.index)
        full_mask.loc[selected_index] = True
        return selected_index, full_mask
    else:
        return selected_index

# ---- usage ----
# selected_idx = select_cd298_high_mhc_low(df, cd298_col="CD298", mhc_col="MHC")
# print(selected_idx)  # pandas Index of the selected cells


In [ ]:
for DB in DBs:
    selected_idx = select_cd298_high_mhc_low(np.arcsinh(globals()[DB]/5), cd298_col="CD298", mhc_col="MHC")
    globals()[DB]=globals()[DB].loc[selected_idx]
    globals()[DB].reset_index(inplace=True)

In [ ]:
N=[
 'BMI1',
 'CD24',

 'CD44',
 'CD45',
 'CD49f',
 'E-cadherin',
 'ER',
 'EpCAM',
 'GATA3',
 'H2AK119ub',
 'H3',
 'H3.3',
 'H3K27ac',
 'H3K27me2',
 'H3K27me3',
 'H3K36me2',
 'H3K36me3',
 'H3K4me1',
 'H3K4me3',
 'H3K64ac',
 'H3K9ac',
 'H3K9me2',
 'H3K9me3',
 'H4',
 'H4K16ac',
 'H4K20me3',
 'KI67',
 'KRT5',
 'KRT8-18',
 'MBD',

 'Pan-KRT',
 'Vimentin',
 'aSMA',
 'pH2A.X',
 'pH3']



In [ ]:
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)
for DB in DBs:
    
    globals()[DB]=globals()[DB][NamesAll]


In [ ]:
for DB in DBs:
    
    sns.histplot(data=np.arcsinh(globals()[DB]/5),x='H3',**hKWD,label=f"{DB}")
plt.legend()
plt.yscale('log')    

In [ ]:
%matplotlib inline

In [ ]:
for DB in DBs:
    plt.figure()
    plt.title(f'{DB}')
    sns.histplot(data=np.arcsinh(globals()[DB]/5),x='H3',**hKWD,color='r')
    sns.histplot(data=np.arcsinh(globals()[DB]/5),x='H3.3',**hKWD,color='g')
    sns.histplot(data=np.arcsinh(globals()[DB]/5),x='H4',**hKWD,color='b')
    
    plt.show()

# Gate on H3.3/H4 too low, but also remove outliers 99.99% from all 

In [ ]:
#C1=C1[(C1.H3K27ME3>0)]

In [ ]:
GateColumns=['H3.3','H4','H3',]
# #Ly7
# print("C01 Ly7")
# print(len(C01),len(C01))
# C01=C01[(C01[GateColumns]>5).all(axis=1)]
# print(len(C01),len(C01))
# C01=C01[(C01<np.quantile(C01,0.9999,axis=0)).all(axis=1)]
# print(len(C01),len(C01))

# #EZH2
# print("C03 EZH2")
# print(len(C03))
# C03=C03[(C03[GateColumns]>5).all(axis=1)]
# print(len(C03))
# C03=C03[(C03<np.quantile(C03,0.9999,axis=0)).all(axis=1)]
# print(len(C03))


def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
#    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data




In [ ]:
len(NamesAll)

In [ ]:
import distinctipy

In [ ]:
CLR=dict(zip(DBs,distinctipy.get_colors(len(DBs))))

In [ ]:
import distinctipy

# Normalize using new method on all intercellular markers

In [ ]:
def R(p,x,data,Q,M,M1,M2,M3):
    a=p['a']
    b=p['b']
    d=x.divide(a*M1+(1-a-b)*M2+b*M3,axis=0)
    return d.std()['H3.3']**2+d.std()['H4']**2+d.std()['H3']**2

def NormalizeNew(data):
    
    params = Parameters()
    params.add('a', value=0.1,min=0,max=1)
    params.add('b', value=0.1,min=0,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4','H3']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
    M3=(ddf/Q)['H3']

    out=minimize(R, params ,args=(ddf, ddf,Q,M,M1,M2,M3),method='cg')
    AA=out.params['a'].value
    BB=out.params['b'].value
    M=M1*AA+M2*(1-AA-BB)+M3*BB
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf
    ddf2[NormMRK]=data[NormMRK]
    data=ddf2.copy()
    print(data.shape,ddf2.shape)

    del ddf 
    del ddf2
    return data
    
def R2(p,x,data,Q,M,M1,M2):
    a=p['a']
    d=x.divide(a*M1+(1-a)*M2,axis=0)
    return (d.std()['H3'])**2+(d.std()['H4'])**2

def NormalizeNew2(data):
    
    params = Parameters()
    params.add('a', value=0.5,min=0.1,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf[NormMRK].mean()
    M=(ddf/Q)[['H3','H4']].mean(axis=1)
    M1=(ddf/Q)['H3']
    M2=(ddf/Q)['H4']
 
    out=minimize(R2, params ,args=(ddf[NormMRK], ddf[NormMRK],Q,M,M1,M2),method='cg')
    AA=out.params['a'].value
    print(AA)
    M=M1*AA+M2*(1-AA)
    ddf[NormMRK]=ddf[NormMRK].divide(M,axis=0).copy()
    data=ddf.copy()
    print(data.shape,ddf2.shape)
    ddf2[NormMRK]=data[NormMRK]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data

In [ ]:
for DB in DBs:

        
    globals()[DB]=NormalizeNew(globals()[DB][NamesAll])


In [ ]:
scFac=5
for DB in DBs:
    globals()[DB]=np.arcsinh(globals()[DB]/scFac)


In [ ]:
EPC=EpiCols.copy()
EPC.remove('H3')
EPC.remove('H4')
EPC.remove('H3.3')

MRK=NamesAll.copy()
MRK.remove('H3')
MRK.remove('H4')
MRK.remove('H3.3')

In [ ]:
for DB in DBs:
    globals()[DB].to_csv(f"Data/{DB}.csv",index=False)

In [ ]:
NC=4000
aaaa=pd.DataFrame(columns=NamesAll)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB][NamesAll].sample(NC,replace=False)
                ]).copy()
                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB][NamesAll]=(globals()[DB][NamesAll]-m)/s


params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")

In [ ]:
UM=umap.UMAP(min_dist=0.001,n_neighbors=160,random_state=42,verbose=True)

In [ ]:
for DB in DBs:
    globals()[DB]['Line']=DB

In [ ]:
CAll=pd.DataFrame(columns=H31_M1_c11.columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB]]).copy()

In [ ]:
EPC.remove('H3K27M')


In [ ]:
Mat=CAll.groupby('Line').mean(numeric_only=True)

In [ ]:
M=Mat.index.str.contains('H31')

In [ ]:
Mat=pd.concat([Mat.loc[M].sort_values(by='H3K27M'),Mat.loc[~M].sort_values(by='H3K27M')])

In [ ]:
sns.heatmap(Mat[EPC+['H3K27M']].T,center=0,cmap=plt.cm.seismic,annot_kws={'fontsize':8},annot=True,yticklabels=True,#col_cluster=True,
              )
plt.yticks(fontsize=8)
plt.savefig("Plots/HM1.png",bbox_inches='tight',dpi=200)

In [ ]:
H31=pd.concat([H31_M1_c11,H31_M2_c12,H31_M3_c13]).copy()

In [ ]:
NMS=['H3K4me1','H3K27me2','H3K27me3']
for N in NMS: 
    plt.figure()
    M,_,_=drevi_matrix(H31['H3K27M'],H31[N],nx=30,ny=30)
    ax,_,_,_=plot_drevi_with_mean(M,cmap='magma_r',xlab='H3K27M',ylab=N);
    plt.savefig(f"Plots/H31_{N}.png",dpi=200,bbox_inches='tight')

In [ ]:
H33=pd.concat([H33_M1_c14,H33_M2_c15,H33_M3_c16]).copy()

In [ ]:
NMS=['H3K4me1','H3K27me2','H3K27me3']
for N in NMS: 
    plt.figure()
    M,_,_=drevi_matrix(H33['H3K27M'],H33[N],nx=30,ny=30)
    ax,_,_,_=plot_drevi_with_mean(M,cmap='magma_r',xlab='H3K27M',ylab=N);
    plt.savefig(f"Plots/H33_{N}.png",dpi=200,bbox_inches='tight')

In [ ]:
CAll=pd.DataFrame(columns=H31_M1_c11.columns)
for DB in DBs:
    CAll=pd.concat([CAll,globals()[DB].sample(5000,replace=False)
                ]).copy()

In [ ]:
sns.histplot(data=CAll,x='H3K27M',hue='Line',**hKWD)

In [ ]:
np.random.seed(2)
UM=umap.UMAP(min_dist=0.001,n_neighbors=10,random_state=42,verbose=True)

In [ ]:
X_2d=UM.fit_transform(CAll[EPC])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1)
plt.show()

In [ ]:
fig,ax = plt.subplots(2,3,figsize=(10,5))
a=ax.flatten()
for i,L in enumerate(CAll.Line.unique()):
   
    M=CAll.Line==L
    
    a[i].scatter(X_2d[M,0],X_2d[M,1],s=1,label=f"{L}",cmap=plt.cm.seismic,c=CLR[L])

    a[i].legend(markerscale=10,fontsize=10)

fig.savefig("Plots/Lines.png",bbox_inches='tight',dpi=200)

In [ ]:
import scanpy as sc
from sknetwork.clustering import Louvain, Leiden

In [ ]:
AN=sc.AnnData(CAll)
AN.obsm['X_umap']=X_2d


In [ ]:
vmn=[]
vmx=[]
for N in MRK:
    cc=CAll[N]
    v1,v2=cc.quantile(0.01),cc.quantile(0.99)
    vmn.append(v1)
    vmx.append(v2)

In [ ]:
sc.pl.umap(AN,color=MRK,cmap='seismic',vmin=vmn,vmax=vmx,show=False)
plt.savefig("Plots/UMAP_All.png",bbox_inches='tight',dpi=200)

In [ ]:
from minisom import MiniSom
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.decomposition import PCA

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(CAll[EPC])

# 3. Initialize & train a 10×10 SOM on the 5-D data
som_size = 30
som = MiniSom(
    x=som_size, y=som_size,
    input_len=len(EPC),
    sigma=1.0, learning_rate=0.5,
    neighborhood_function='gaussian',
    random_seed=42
)
som.random_weights_init(X_scaled)
print("Training SOM on 5-D car data…")
som.train_random(X_scaled, num_iteration=20000)

# 4. Compute and plot the U-matrix
umatrix = som.distance_map().T
plt.figure(figsize=(6,6))
plt.pcolor(umatrix, cmap='bone_r')
plt.colorbar(label='U-matrix (avg neighbor distance)')
plt.title("SOM U-Matrix")
plt.show()


In [ ]:
codebook = som.get_weights().reshape(-1, X_scaled.shape[1])  # (100, 5)
n_clusters = 6
km = KMeans(n_clusters=n_clusters, random_state=0).fit(codebook)
node_labels = km.labels_.reshape(som_size, som_size)

# 6. Assign each car to its BMU’s cluster
bmus = np.array([som.winner(x) for x in X_scaled])
data_labels = np.array([node_labels[i,j] for (i,j) in bmus])

# 7. Visualize clusters in PCA space for interpretability

scatter = plt.scatter(
    X_2d[:,0], X_2d[:,1],
    c=data_labels, cmap='tab10',
    s=2
)
plt.legend(*scatter.legend_elements(), title="Cluster")
plt.tight_layout()
plt.show()


In [ ]:
for L in CAll.Line.unique():
    M1=CAll.Line==L
    print(f"Line {L}")
    for i in range(np.max(lbl)+1):
        M2=M1 & (CAll.Cl==i)
        print(f"Cluster {i} : {np.round(M2.sum()/M1.sum(),4)}")

    print("\n")

In [ ]:
M=CAll.Line=='H3.3'

In [ ]:
CAll2=CAll[M].copy()

In [ ]:
m=CAll2.mean(numeric_only=True,axis=0)
s=CAll2.std(numeric_only=True,axis=0)
CAll2=(CAll2-m)/s

In [ ]:
CAll2['Line']='H3.3'

In [ ]:
UM=umap.UMAP(min_dist=0.001,n_neighbors=50,random_state=42,verbose=True)

In [ ]:
X_2d=UM.fit_transform(CAll2[EPC])
plt.scatter(X_2d[:,0],X_2d[:,1],s=1)
plt.show()

In [ ]:
fig,ax = plt.subplots(1,3,figsize=(10,5))
a=ax.flatten()
for i,L in enumerate(CAll2.Line.unique()):
   
    M=CAll2.Line==L
    
    a[i].scatter(X_2d[M,0],X_2d[M,1],s=1,label=f"{L}",cmap=plt.cm.seismic,c=DBCLR[L])

    a[i].legend(markerscale=10,fontsize=10)



In [ ]:
AN=sc.AnnData(CAll2)
AN.obsm['X_umap']=X_2d


In [ ]:
vmn=[]
vmx=[]
for N in MRK:
    cc=CAll[N]
    v1,v2=cc.quantile(0.01),cc.quantile(0.99)
    vmn.append(v1)
    vmx.append(v2)

In [ ]:
sc.pl.umap(AN,color=MRK,cmap='seismic',vmin=vmn,vmax=vmx,show=False)
plt.savefig("Plots/UMAP_H33.png",bbox_inches='tight',dpi=200)

In [ ]:
plt.figure(figsize=(10,10))
sns.clustermap(CAll2[EPC+['p53']].corr(),annot=True,annot_kws={'fontsize':8},
               xticklabels=True, yticklabels=True,
               cmap=plt.cm.seismic,center=0)
plt.savefig("Plots/Corr_H33.png",bbox_inches='tight',dpi=200)

In [ ]:
lbl=Leiden(resolution=0.2).fit_predict(UM.graph_)
NC=np.max(lbl)+1
print(NC)

In [ ]:
lbl=DBSCAN(eps=0.4,min_samples=40).fit_predict(X_2d)
NC=np.max(lbl)+1
print(NC)

In [ ]:
CLR=dict(zip(range(NC),distinctipy.get_colors(NC)))
plt.scatter(X_2d[:,0],X_2d[:,1],s=1,color='gray')
for i in range(NC):
    M=lbl==i
    plt.scatter(X_2d[M,0],X_2d[M,1],s=1,color=CLR[i],label=f'Cluster {i}')

plt.legend(bbox_to_anchor=(1,1),markerscale=10)
#plt.savefig("Plots/UMAP_CI_IDX.png",bbox_inches='tight',dpi=200)

In [ ]:
CAll2['Cl']=lbl

In [ ]:
Mat=CAll2[CAll2.Cl>-1].groupby('Cl').mean(numeric_only=True)

In [ ]:
g=sns.clustermap(Mat[MRK].T,center=0,cmap=plt.cm.seismic,annot_kws={'fontsize':8},annot=True,
           yticklabels=True,col_cluster=True,)
plt.yticks(fontsize=8);
plt.savefig("Plots/HM_H33.png",bbox_inches='tight',dpi=200)